In [ ]:
import torch
import torch.nn as nn
from model.msg.msg import  MSGNet
from model.espcn import ESPCN
from model.srx8 import SYESRX8NetS

In [ ]:
class testnet(nn.Module):
    def __init__(
        self,
    ):
        super(testnet,self).__init__()
        self.conv1 = nn.Conv2d(3, 1, 1)
        self.prompt_conv = nn.Conv2d(3,1,1)
    def forward(self,x):
        return x

In [ ]:
ckpt = torch.load('./checkpoint_step-80000')
msg = MSGNet(8).to('cuda:1')
msg.load_state_dict(ckpt['model'])
escpn = ESPCN().to('cuda:1')
escpn.load_state_dict(torch.load('/home/ziyao/syenet/experiments/2024-03-08 15-59-06 train_ESCPNsr/models/model_best.pkl'))
sye = SYESRX8NetS(36)
sye.load_state_dict(torch.load('/home/ziyao/syenet/experiments/2024-03-08 15-57-36 train_syesr/models/model_best.pkl'))

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as T
import torch.nn.functional as F
transform = T.ToPILImage()
depth_dir = '/home/ziyao/ArkitData/upsampling/Validation/lowres_depth/44358451_43682.743.png'
depth= Image.open(depth_dir)
depth = np.array(depth)
depth = depth.astype(np.float32) / 1000
depth = depth/ np.max(depth)
depth = torch.Tensor(np.array(depth)).unsqueeze(0).unsqueeze(0)
with torch.no_grad():
    depth_sr = sye(depth)
depth_bi = F.interpolate(depth,scale_factor=8).squeeze()
depth_sr  = depth_sr.squeeze()
depth_diff = torch.absolute(depth_sr - depth_bi)
depth_save = transform(depth_sr)
depth_bi = transform(depth_bi)
depth_diff_save = transform(depth_diff)
# depth_save.save('depth_sr_1.pdf')
# depth_bi.save('depth_bi_1.pdf')
# depth_diff_save.save('depth_diff_1.pdf')
depth_sr = depth_sr.squeeze().numpy()
dpeth_sr = depth_sr
depth_diff = depth_diff.squeeze().numpy()
plt.subplot(1,2,1)
plt.imshow(depth_sr)
plt.subplot(1,2,2)
plt.imshow(depth_diff)


In [ ]:
from depth_anything.dpt import DepthAnything
import torch.nn as nn
import torch
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as T
import torch.nn.functional as F
transform = T.ToPILImage()
class DepthEstimateLoss(nn.Module):
    def __init__(self,device):
        super(DepthEstimateLoss,self).__init__()
        self.depth_anything = DepthAnything.from_pretrained('LiheYoung/depth_anything_{}14'.format('vits')).to(device).eval()
        for param in self.depth_anything.parameters():
            param.requires_grad = False
    
    def depth_estimate(self, img):
        img = img.unsqueeze(0)
        b,c,h,w = img.size()
        size = (140,140) if h < 518 else (518,518)
        img = F.interpolate(img,size=size)
        depth = self.depth_anything(img)
        depth = F.interpolate(depth[None], (h, w), mode='bilinear', align_corners=False)

        depth = (depth - depth.min()) / (depth.max() - depth.min())
        
        return (1 - depth).squeeze(0)

depth_est = DepthEstimateLoss("cpu")
blur = '/home/ziyao/ArkitData/upsampling/Validation/wide/47332005_172100.816.png'
blur = Image.open(blur)
blur = np.array(blur)
blur = blur.transpose(2, 0, 1)
blur = blur.astype(np.float32) / 255
blur = torch.Tensor(blur)
depth = depth_est.depth_estimate(blur)
depth = transform(depth)
depth.save('./bathroom_est_clear.pdf')
plt.imshow(depth)
plt.axis("off")

In [ ]:
def cal_psnr(img,gt):
    high_depth = gt[:,0,:,:].unsqueeze(1)
    mask = gt[:,1,:,:].unsqueeze(1)
    mse = torch.sum(((img - high_depth)*mask)**2)/torch.sum(mask)
    psnr = (1 / mse).log10().mean() * 10
    return psnr

In [ ]:
import torch
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import os
import torch.nn.functional as F
sye = SYESRX8NetS(36)
sye.load_state_dict(torch.load('/home/ziyao/syenet/experiments/2024-03-08 15-57-36 train_syesr/models/model_best.pkl'))
depth_dir = '/home/ziyao/ArkitData/upsampling/Validation/lowres_depth'
for file in os.listdir(depth_dir):
    depth= Image.open(os.path.join(depth_dir,file))
    depth = np.array(depth)
    depth = depth.astype(np.float32) / 1000
    depth = depth/ np.max(depth)
    depth = torch.Tensor(np.array(depth)).unsqueeze(0).unsqueeze(0)
    with torch.no_grad():
        depth_sr = sye(depth)
    depth_bi = F.interpolate(depth, scale_factor=8)
    err = F.l1_loss(depth_sr,depth_bi)
    if err>0.009:
        print(file,err)

In [ ]:
import torch
import torch.nn as nn
from model.msg.msg import  MSGNet
from model.espcn import ESPCN
from model.srx8 import SYESRX8NetS
import torchvision.transforms as T
from PIL import Image
import os
transform = T.ToPILImage()
sye = SYESRX8NetS(36)
depth_dir = '/home/ziyao/ArkitData/upsampling/Validation/lowres_depth'
depth_list = os.listdir('/home/ziyao/ArkitData/upsampling/Validation/visual_test')
depth= Image.open(depth_dir)
depth = np.array(depth)
depth = depth.astype(np.float32) / 1000
depth = depth/ np.max(depth)
depth = torch.Tensor(np.array(depth)).unsqueeze(0).unsqueeze(0)
sye.load_state_dict(torch.load('/home/ziyao/syenet/experiments/2024-03-08 15-57-36 train_syesr/models/model_best.pkl'))
with torch.no_grad():
    depth_sr = sye(depth)
depth_sr = depth_sr.squeeze().squeeze().numpy()
dpeth_sr = depth_sr
depth_sr = transform(depth_sr)
depth_sr.save('./depth_tensor/'+filename)

In [ ]:

low_depth_path ='/home/ziyao/ArkitData/upsampling/Validation/lowres_depth/47332005_172100.816.png'
high_depth_path = '/home/ziyao/ArkitData/upsampling/Validation/highres_depth/47332005_172100.816.png'
rgb = '/home/ziyao/ArkitData/upsampling/Validation/wide/47332005_172100.816.png'
blur = '/home/ziyao/ArkitData/upsampling/Validation/small/47332005_172100.816.png'
from PIL import Image
from depth_anything

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import math
depth_rgb = Image.open("/home/ziyao/deblurvit/experiments/2024-06-04_17-30-05_test_prompt_restormer/images/47332908_12131.719.png")
depth_rgb = np.array(depth_rgb)
rgb = Image.open("/home/ziyao/deblurvit/experiments/2024-06-04_16-48-11_test_Deblur_restormer/images/47332908_12131.719.png")
rgb = np.array(rgb)
gt = Image.open("/home/ziyao/ArkitData/upsampling/Validation/wide/47332908_12131.719.png")
gt = np.array(gt)

residual_depth = np.absolute(gt-depth_rgb)
plt.subplot(1,2,1)
plt.imshow(residual_depth)
plt.axis("off")
residual_rgb = np.absolute(gt-rgb)
plt.subplot(1,2,2)
plt.imshow(residual_rgb)
plt.axis("off")

In [ ]:
from PIL import Image
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt

low_depth = Image.open('/home/ziyao/ArkitData/upsampling/Validation/small_depth/41069042_3096.900.png')
low_depth = np.array(low_depth)
low_depth = low_depth.astype(np.float32) / 1000
low_depth = torch.Tensor(low_depth).to('cuda:1').unsqueeze(0).unsqueeze(0)
print(low_depth.size())
fig = plt.figure(figsize=(20,10))
rgb = Image.open('/home/ziyao/ArkitData/upsampling/Validation/blurred/41069042_3096.900_ker3.png')
fig.add_subplot(2,5,1,title='blur_image')
plt.imshow(rgb)
rgb_c = Image.open('/home/ziyao/ArkitData/upsampling/Validation/wide/41069042_3096.900.png')
rgb = np.array(rgb)
rgb = rgb.transpose((2,0,1))
rgb = rgb.astype(np.float32) / 255
rgb = torch.Tensor(rgb).to('cuda:1').unsqueeze(0)
fig.add_subplot(2,5,6,title='clear_image')
plt.imshow(rgb_c)
rgb_c = np.array(rgb_c)
rgb_c = rgb_c.transpose((2,0,1))
rgb_c = rgb_c.astype(np.float32) / 255
rgb_c = torch.Tensor(rgb_c).to('cuda:1').unsqueeze(0)
high_depth = Image.open('/home/ziyao/ArkitData/upsampling/Validation/highres_depth/41069042_3096.900.png')
high_depth = np.array(high_depth)
fig.add_subplot(2,5,4,title='highresp_depth')
plt.imshow(high_depth/1000)
fig.add_subplot(2,5,9,title='highresp_depth')
plt.imshow(high_depth/1000)
mask = np.where(high_depth>0.01,1,0)
mask = torch.Tensor(mask).to('cuda:1').unsqueeze(0)
high_depth = high_depth.astype(np.float32) / 1000
high_depth = torch.Tensor(high_depth).to('cuda:1').unsqueeze(0)
high_depth_n = high_depth / high_depth.max()
low_depth_n = low_depth / low_depth.max()
mask = torch.Tensor(mask).to('cuda:1').unsqueeze(0)
h,c,h,w = rgb.size()
low_depth = F.interpolate(low_depth, size=(h//8,w//8))
low_depth_m = F.interpolate(low_depth_n, size=(h//8,w//8))
with torch.no_grad():
    out = msg(low_depth,rgb)
    fig.add_subplot(2,5,2,title='blurred_img_guided_upsample')
    plt.imshow(out.cpu().squeeze(0,1).numpy())
    out_c = msg(low_depth,rgb_c)
    mse = torch.sum(((out_c - high_depth)*mask)**2)/torch.sum(mask)
    psnr_clear = (1 / mse).log10().mean() * 10
    fig.add_subplot(2,5,7,title='clear_img_guided_upsample')
    plt.imshow(out_c.cpu().squeeze(0,1).numpy())
    out_e = sye(low_depth_m)
    mse = torch.sum(((out_e - high_depth_n)*mask)**2)/torch.sum(mask)
    psnr_ours = (1 / mse).log10().mean() * 10
    fig.add_subplot(2,5,3,title='sye_upsample')
    plt.imshow(out_e.cpu().squeeze(0,1).numpy())
    fig.add_subplot(2,5,8,title='sye_upsample')
    plt.imshow(out_e.cpu().squeeze(0,1).numpy())
    mse = torch.sum(((out - high_depth)*mask)**2)/torch.sum(mask)
    psnr_blur = (1 / mse).log10().mean() * 10
    fig.add_subplot(2,5,5, title='iphone_lowres_depth')
    plt.imshow(low_depth.squeeze(0,1).cpu().numpy())
    fig.add_subplot(2,5,10,title='iphone_lowres_depth')
    plt.imshow(low_depth.squeeze(0,1).cpu().numpy())
    bi =  F.interpolate(low_depth, size=(h,w))
    mse = torch.sum(((bi - high_depth)*mask)**2)/torch.sum(mask)
    psnr_bi = (1 / mse).log10().mean() * 10
print('apple_blur: ',psnr_blur.item(), 'apple_clear: ', psnr_clear.item(), 'bicubic', psnr_bi.item(), 'ours: ', psnr_ours.item())

In [ ]:
import cv2
from scipy.io import loadmat

rgb_c = Image.open('/home/ziyao/ArkitData/upsampling/Validation/wide/41069042_3096.900.png')
def generate_blur(rgb):
    ker_mat = loadmat("/home/ziyao/ArkitData/Levin09_v7.mat")
    ker_mat = ker_mat['kernels']
    get_ker = lambda idx: ker_mat[0, idx]
    ker = get_ker(5)
    h, w, _ = rgb.shape
    scale = 1440/255
    hk, wk = ker.shape
    ker = cv2.resize(ker, (int(hk * scale), int(wk * scale)), interpolation=cv2.INTER_LINEAR)
    ker /= ker.sum()
    blur = cv2.filter2D(rgb, -1, ker)
    return blur

In [ ]:

rgb_c = Image.open('/home/ziyao/ArkitData/upsampling/Validation/wide/41069042_3096.900.png')
rgb_c = np.array(rgb_c)
blur = generate_blur(rgb_c)
rgb_c_patch = rgb_c[250:250+128, 1250:1250+128,:]
blur_patch = generate_blur(rgb_c_patch)
fig2 = plt.figure(figsize=(10,5))
fig2.add_subplot(1,2,1,title='blur_first')
plt.imshow(blur[250:250+128, 1250:1250+128,:])
fig2.add_subplot(1,2,2,title='cut_first')
plt.imshow(blur_patch)

In [ ]:
import random
def get_patch_pair(dep, gt, patch_size):
    hb,wb,_ = gt.shape
    hd,wd = dep.shape
    scale = int(hb// hd)
    dep_h, dep_w = dep.shape # c h w
    dep_patch_size = int(patch_size// scale *2)
    dep_h_start = random.randint(0, dep_h - dep_patch_size)
    dep_w_start = random.randint(0, dep_w - dep_patch_size)
    print(dep_h_start,dep_w_start)
    dep_patch = dep[dep_h_start: dep_h_start + dep_patch_size, dep_w_start: dep_w_start + dep_patch_size]
    gt_patch_size = int(patch_size *2)
    gt_h_start = int(dep_h_start * scale)
    gt_w_start = int(dep_w_start * scale)
    print(gt_h_start,gt_w_start)
    gt_patch = gt[gt_h_start: gt_h_start + gt_patch_size, gt_w_start: gt_w_start + gt_patch_size,:]
    # blur_patch = blur[:, gt_h_start: gt_h_start + gt_patch_size, gt_w_start: gt_w_start + gt_patch_size]

    return  dep_patch, gt_patch


In [ ]:
import cv2
def get_patch_pair2(dep, gt, patch_size):
    hb,wb,_ = gt.shape
    scale = 8
    dep = cv2.resize(dep,(wb//8,hb//8))
    dep_h, dep_w = dep.shape # c h w
    dep_patch_size = int(patch_size// scale *2)
    dep_h_start = random.randint(0, dep_h - dep_patch_size)
    dep_w_start = random.randint(0, dep_w - dep_patch_size)
    dep_patch = dep[dep_h_start: dep_h_start + dep_patch_size, dep_w_start: dep_w_start + dep_patch_size]
    gt_patch_size = int(patch_size *2)
    gt_h_start = int(dep_h_start * scale)
    gt_w_start = int(dep_w_start * scale)
    gt_patch = gt[gt_h_start: gt_h_start + gt_patch_size, gt_w_start: gt_w_start + gt_patch_size,:]
    # blur_patch = blur[:, gt_h_start: gt_h_start + gt_patch_size, gt_w_start: gt_w_start + gt_patch_size]

    return  dep_patch, gt_patch

In [ ]:
from PIL import Image
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
rgb_c = Image.open('/home/ziyao/ArkitData/upsampling/Validation/wide/41069043_2835.908.png')
dep = Image.open('/home/ziyao/ArkitData/upsampling/Validation/lowres_depth/41069043_2835.908.png')
rgb_c = np.array(rgb_c)
dep = np.array(dep) /np.max(dep)

In [ ]:
print(rgb_c.shape, dep.shape)
patch_size = 128
dep_patch, rgb_patch = get_patch_pair2(dep, rgb_c, patch_size)
print(rgb_patch.shape, dep_patch.shape)
plt.subplot(2,2,1)
plt.imshow(rgb_patch[patch_size//2: patch_size//2 + patch_size, patch_size//2: patch_size//2 + patch_size,:])
plt.subplot(2,2,2)
dh,dw = dep_patch.shape
plt.imshow(dep_patch[dh // 4 - 1: int( dh // 4 * 3),  dw  // 4 - 1: int(dw // 4 * 3)])
plt.subplot(2,2,3)
plt.imshow(rgb_patch)
plt.subplot(2,2,4)
plt.imshow(dep_patch)


In [ ]:
sye2 = SYESRX8NetS(36).to('cuda:1')
sye2.load_state_dict(torch.load('./srx8.pkl'))

In [ ]:
param1 = list(sye.parameters())
param2 = list(sye2.parameters())
for p1, p2 in zip(param1,param2):
    if not torch.allclose(p1.data, p2.data):
        print('not equal')
        break

In [ ]:
import ptwt
import torch
import torch.nn as nn 
from PIL import Image
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
rgb_c = Image.open('/home/ziyao/ArkitData/upsampling/Validation/wide/41069042_3096.900.png')
rgb = np.array(rgb_c)
rgb_tensor = torch.Tensor(np.array(rgb_c)).unsqueeze(0)
rgb_tensor = torch.cat((rgb_tensor,rgb_tensor),dim=0)

In [ ]:
import ptwt, pywt
rgb_tensor = rgb_tensor.permute(0,3,1,2)
print(rgb_tensor.size())
coefficients = ptwt.wavedec2(rgb_tensor,pywt.Wavelet("haar"), level=2)
# print(coefficients[1][2].size())
# coeff2 = pywt.dwt2(rgb, 'haar')
# print(coeff2[1][0].shape)
LL, (LH2,HL2,HH2),(LH1,HL1,HH1) = coefficients
print(LL.size(),LH1.size(), LH2.size())

In [ ]:
from pytorch_wavelets import DWTForward, DWTInverse
xfm = DWTForward(J=1, mode='zero', wave='db3')  # Accepts all wave types available to PyWavelets
ifm = DWTInverse(mode='zero', wave='db3')
Yl, Yh = xfm(rgb_tensor)
Yh[0].size()

In [7]:
from thop import profile
import torch
from model.Eeffdepunetwave import EfficientDepUnetWave
from model.effunetwave import EfficientUnetWave
from model.effdepunet import EfficientDepUnet
from model.restormer import Restormer
from model.archs.NAFNet_arch import NAFNet
from model.depthpromptrestormer import DepthPromptRestormer
from model.depthDeblurDiNATL import DepthNADeblurL
from model.DeblurDiNATL import NADeblurL
from model.stripformer import Stripformer
from model.depthstripformer import DepthStripformer
# eff = EfficientDepUnetWave(middle_channel=16)
# naf = NAFNet( width=64, enc_blk_nums=[1, 1, 1, 28],middle_blk_num=1,dec_blk_nums=[1, 1, 1, 1])
# rest = Restormer()
# deprest = DepthPromptRestormer()
model = DepthStripformer().to('cuda')
input = torch.randn(1,3,1440,1920).to('cuda')
dep = torch.randn(1,1,144,256).to('cuda')
macs, params = profile(model, inputs=(input,dep))
print(macs/1e9, params/1e6)

[INFO] Register count_relu() for <class 'torch.nn.modules.activation.LeakyReLU'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register count_softmax() for <class 'torch.nn.modules.activation.Softmax'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register count_prelu() for <class 'torch.nn.modules.activation.PReLU'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.pixelshuffle.PixelShuffle'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.ConvTranspose2d'>.


/home/ziyao/miniconda3/envs/deblur/lib/python3.10/site-packages/thop/vision/calc_func.py:53: UserWarning: This API is being deprecated
  warnings.warn("This API is being deprecated")


9753.051103848 22.240585


In [ ]:
from thop import profile
import torch
from model.Eeffdepunetwave import EfficientDepUnetWave
import ptwt, pywt
from model.effunetwave import EfficientUnetWave
from model.effdepunet import EfficientDepUnet
from model.restormer import Restormer
from model.archs.NAFNet_arch import NAFNet
from model.effdepthwavenew import EfficientDepUnetWaveNEW
import time
fasteff = EfficientDepUnetWaveNEW(middle_channel=16).to('cuda:1')
eff = EfficientDepUnetWave(middle_channel = 16).to('cuda:1')
eff_pure = EfficientUnetWave(middle_channel=16).to('cuda:1')
naf = NAFNet( width=32, enc_blk_nums=[1, 1, 1, 28],middle_blk_num=1,dec_blk_nums=[1, 1, 1, 1]).to('cuda:1')
input = torch.randn(1,3,1440,1920).to('cuda:1')
dep = torch.randn(1,1,144,256).to('cuda:1')
naf_time = []
fasteff_time = []
eff_time = []
with torch.no_grad():
    for i in range(10):
    # dep = torch.randn(1,1,144,256).to('cuda')
        a = time.time()
        out = fasteff(input,dep)
        b = time.time()
        fasteff_time.append(b-a)
        c = time.time()
        out2 = eff_pure(input)
        d = time.time()
        eff_time.append(d-c)
        e = time.time()
        out3 = naf(input)
        f = time.time()
        naf_time.append(f-e)
print('new wavelet: {}, wavelet: {}, naf: {}'.format(sum(fasteff_time)/10, sum(eff_time)/10, sum(naf_time)/10))


In [ ]:
from thop import profile
import torch
from model.Eeffdepunetwave import EfficientDepUnetWave
import ptwt, pywt
from model.effunetwave import EfficientUnetWave
from model.effdepunet import EfficientDepUnet
from model.restormer import Restormer
from model.archs.NAFNet_arch import NAFNet
from model.effdepthwavenew import EfficientDepUnetWaveNEW
import time
device ='cuda:1'
fasteff = EfficientDepUnetWaveNEW(middle_channel=16).to(device)
eff = EfficientDepUnetWave(middle_channel = 16).to(device)
eff_pure = EfficientUnetWave(middle_channel=16).to(device)
naf = NAFNet( width=32, enc_blk_nums=[1, 1, 1, 28],middle_blk_num=1,dec_blk_nums=[1, 1, 1, 1]).to(device)
input = torch.randn(1,3,1440,1920).to(device)
dep = torch.randn(1,1,144,256).to(device)
naf_time = []
fasteff_time = []
eff_time = []
with torch.no_grad():
    for i in range(10):
    # dep = torch.randn(1,1,144,256).to('cuda')
        a = time.time()
        out = fasteff(input,dep)
        b = time.time()
        fasteff_time.append(b-a)
print('new wavelet: {}'.format(sum(fasteff_time)/10))

In [ ]:
import matplotlib.pyplot as plt
import math
x_idx = [math.log10(540 * 1e9), math.log10(2132 * 1e9)]
y_idx = [35.01, 35.50]
x_idx2 = [math.log10(673 * 1e9), math.log10(2661 * 1e9)]
y_idx2 = [35.63,36.95]
fig,ax1 = plt.subplots()
ax1.plot(x_idx, y_idx,'b',label='Effuent')
ax1.plot(x_idx2, y_idx2,'r',label='NAFnet')
label = ['Effunet','Nafunet']
plt.xlabel('log on MACs')
plt.ylabel('PSNR')
fig.legend(label)

In [ ]:
from model.effdepthwavenew import EfficientDepUnetWaveNEW
from model.archs.NAF_depth import NAFNet_depth
from model.archs.NAFNet_arch import NAFNet
from PIL import Image
import numpy as np
import torch
import matplotlib.pyplot as plt
blur_img = Image.open('/home/ziyao/dreamclean/panda.png')
blur = np.array(blur_img).transpose([2, 0, 1]) 
blur = blur.astype(np.float32) / 255
blur = torch.Tensor(blur).unsqueeze(0)
depth= Image.open('/home/ziyao/dreamclean/panda_depth.tiff')
depth = np.array(depth)
depth = depth.astype(np.float32) / np.max(depth)
depth = torch.Tensor(np.array(depth)).unsqueeze(0).unsqueeze(0)
fasteff = EfficientDepUnetWaveNEW(middle_channel=16)
fasteff.load_state_dict(torch.load('/home/ziyao/deblurvit/experiments/2024-05-14_12-22-38_train_depth_new_effiecient_epoch400/models/model_best.pth',map_location='cpu')['model_state_dict'],strict=True)
nafdep = NAFNet_depth(width=32, enc_blk_nums=[1, 1, 1, 28],middle_blk_num=1,dec_blk_nums=[1, 1, 1, 1])
nafdep.load_state_dict(torch.load('/home/ziyao/deblurvit/experiments/2024-05-02_11-38-40_train_depth_NAFNet_32_epoch50/models/model_best.pth',map_location='cpu')['model_state_dict'],strict=True)
naf = NAFNet(width=32, enc_blk_nums=[1, 1, 1, 28],middle_blk_num=1,dec_blk_nums=[1, 1, 1, 1])
naf.load_state_dict(torch.load('/home/ziyao/deblurvit/experiments/2024-05-03_10-31-01_train_Deblur_NAFNet32_width32_epoch100/models/model_best.pth',map_location='cpu')['model_state_dict'],strict=True)
with torch.no_grad():
    out_1 = fasteff(blur,depth)
out_1 = (out_1.clip(0, 1)[0] * 255).permute([1, 2, 0]).numpy().astype(np.uint8)
with torch.no_grad():
    out_2 = nafdep(blur,depth)
out_2 = (out_2.clip(0, 1)[0] * 255).permute([1, 2, 0]).numpy().astype(np.uint8)
with torch.no_grad():
    out_3 = naf(blur)
out_3 = (out_3.clip(0, 1)[0] * 255).permute([1, 2, 0]).numpy().astype(np.uint8)
plt.figure(figsize=(6*3, 6))
plt.subplot(1,4,1)
plt.imshow(blur_img)
plt.axis('off')
plt.title('blur')
plt.subplot(1,4,2)
plt.imshow(out_1)
plt.axis('off')
plt.title('effdepth')
plt.subplot(1,4,3)
plt.imshow(out_2)
plt.axis('off')
plt.title('nafdepth')
plt.subplot(1,4,4)
plt.imshow(out_3)
plt.axis('off')
plt.title('naf')

In [ ]:
from model.effdepthwavenew import EfficientDepUnetWaveNEW
from model.archs.NAF_depth import NAFNet_depth
from model.archs.NAFNet_arch import NAFNet
from PIL import Image
import numpy as np
import torch
import matplotlib.pyplot as plt
blur_img = Image.open('/home/ziyao/dreamclean/panda_less.png')
blur = np.array(blur_img).transpose([2, 0, 1]) 
blur = blur.astype(np.float32) / 255
blur = torch.Tensor(blur).unsqueeze(0)
depth= Image.open('/home/ziyao/dreamclean/panda_depth_less.tiff')
depth = np.array(depth)
depth = depth.astype(np.float32) / np.max(depth)
depth = torch.Tensor(np.array(depth)).unsqueeze(0).unsqueeze(0)
fasteff = EfficientDepUnetWaveNEW(middle_channel=16)
fasteff.load_state_dict(torch.load('/home/ziyao/deblurvit/experiments/2024-05-14_12-22-38_train_depth_new_effiecient_epoch400/models/model_best.pth',map_location='cpu')['model_state_dict'],strict=True)
nafdep = NAFNet_depth(width=32, enc_blk_nums=[1, 1, 1, 28],middle_blk_num=1,dec_blk_nums=[1, 1, 1, 1])
nafdep.load_state_dict(torch.load('/home/ziyao/deblurvit/experiments/2024-05-02_11-38-40_train_depth_NAFNet_32_epoch50/models/model_best.pth',map_location='cpu')['model_state_dict'],strict=True)
naf = NAFNet(width=32, enc_blk_nums=[1, 1, 1, 28],middle_blk_num=1,dec_blk_nums=[1, 1, 1, 1])
naf.load_state_dict(torch.load('/home/ziyao/deblurvit/experiments/2024-05-03_10-31-01_train_Deblur_NAFNet32_width32_epoch100/models/model_best.pth',map_location='cpu')['model_state_dict'],strict=True)
with torch.no_grad():
    out_1 = fasteff(blur,depth)
out_1 = (out_1.clip(0, 1)[0] * 255).permute([1, 2, 0]).numpy().astype(np.uint8)
with torch.no_grad():
    out_2 = nafdep(blur,depth)
out_2 = (out_2.clip(0, 1)[0] * 255).permute([1, 2, 0]).numpy().astype(np.uint8)
with torch.no_grad():
    out_3 = naf(blur)
out_3 = (out_3.clip(0, 1)[0] * 255).permute([1, 2, 0]).numpy().astype(np.uint8)
plt.figure(figsize=(6*3, 6))
plt.subplot(1,4,1)
plt.imshow(blur_img)
plt.axis('off')
plt.title('blur')
plt.subplot(1,4,2)
plt.imshow(out_1)
plt.axis('off')
plt.title('effdepth')
plt.subplot(1,4,3)
plt.imshow(out_2)
plt.axis('off')
plt.title('nafdepth')
plt.subplot(1,4,4)
plt.imshow(out_3)
plt.axis('off')
plt.title('naf')

In [ ]:
from model.effdepthwavenew import EfficientDepUnetWaveNEW
from model.archs.NAF_depth import NAFNet_depth
from model.archs.NAFNet_arch import NAFNet
from PIL import Image
import numpy as np
import torch
import matplotlib.pyplot as plt
# /home/ziyao/ArkitData/upsampling/Validation/small_set/42445028_49647.065.png  41069050_5057.295.png
blur_img = Image.open('/home/ziyao/ArkitData/upsampling/Validation/small_set/42445028_49647.065.png')
blur = np.array(blur_img).transpose([2, 0, 1]) 
blur = blur.astype(np.float32) / 255
blur = torch.Tensor(blur).unsqueeze(0)
depth= Image.open('/home/ziyao/ArkitData/upsampling/Validation/lowres_depth/41069050_5057.295.png')
depth = np.array(depth)
depth = depth.astype(np.float32) / np.max(depth)
depth = torch.Tensor(np.array(depth)).unsqueeze(0).unsqueeze(0)
fasteff = EfficientDepUnetWaveNEW(middle_channel=16)
fasteff.load_state_dict(torch.load('/home/ziyao/deblurvit/experiments/2024-05-14_12-22-38_train_depth_new_effiecient_epoch400/models/model_best.pth',map_location='cpu')['model_state_dict'],strict=True)
nafdep = NAFNet_depth(width=32, enc_blk_nums=[1, 1, 1, 28],middle_blk_num=1,dec_blk_nums=[1, 1, 1, 1])
nafdep.load_state_dict(torch.load('/home/ziyao/deblurvit/experiments/2024-05-02_11-38-40_train_depth_NAFNet_32_epoch50/models/model_best.pth',map_location='cpu')['model_state_dict'],strict=True)
naf = NAFNet(width=32, enc_blk_nums=[1, 1, 1, 28],middle_blk_num=1,dec_blk_nums=[1, 1, 1, 1])
naf.load_state_dict(torch.load('/home/ziyao/deblurvit/experiments/2024-05-03_10-31-01_train_Deblur_NAFNet32_width32_epoch100/models/model_best.pth',map_location='cpu')['model_state_dict'],strict=True)
with torch.no_grad():
    out_1 = fasteff(blur,depth)
out_1 = (out_1.clip(0, 1)[0] * 255).permute([1, 2, 0]).numpy().astype(np.uint8)
with torch.no_grad():
    out_2 = nafdep(blur,depth)
out_2 = (out_2.clip(0, 1)[0] * 255).permute([1, 2, 0]).numpy().astype(np.uint8)
with torch.no_grad():
    out_3 = naf(blur)
out_3 = (out_3.clip(0, 1)[0] * 255).permute([1, 2, 0]).numpy().astype(np.uint8)
plt.figure(figsize=(6*3, 6))
plt.subplot(1,4,1)
plt.imshow(blur_img)
plt.axis('off')
plt.title('blur')
plt.subplot(1,4,2)
plt.imshow(out_1)
plt.axis('off')
plt.title('effdepth')
plt.subplot(1,4,3)
plt.imshow(out_2)
plt.axis('off')
plt.title('nafdepth')
plt.subplot(1,4,4)
plt.imshow(out_3)
plt.axis('off')
plt.title('naf')

In [1]:
import model
import os
import torch 
import numpy as np
from PIL import Image
import cv2

import model.archs
import model.archs.NAFNet_arch
blur_path = '/home/ziyao/ArkitData/upsampling/Validation/visual_test/'
depth_path = '/home/ziyao/ArkitData/upsampling/Validation/lowres_depth/'
rgb_path = "/home/ziyao/ArkitData/upsampling/Validation/wide/"
device = 'cuda'

model = model.DepthPromptRestormer()
model = model.to(device)
model.load_state_dict(torch.load(' /home/ziyao/deblurvit/experiments/2024-03-14_23-15-44_train_Deblur_Restormer_finetuning_small_set/models/model_best.pth',map_location=device)['model_state_dict'], strict=True)
model_name = "Restormer.pdf"
for file in os.listdir(blur_path):
    save_path = os.path.join('./paperfig', file.split('.png')[0])
    blur = Image.open(os.path.join(blur_path,file))
    depth = Image.open(os.path.join(depth_path,file))
    blur = np.array(blur).transpose([2, 0, 1])
    blur = blur.astype(np.float32) / 255
    blur = torch.Tensor(np.array(blur)).unsqueeze(0)
    blur = blur.to(device)
    depth = np.array(depth)
    depth = depth.astype(np.float32) / np.max(depth)
    depth = torch.Tensor(np.array(depth)).unsqueeze(0).unsqueeze(0)
    depth = depth.to(device)
    with torch.no_grad():
        out = model(blur,depth)
        img = (out.clip(0, 1)[0] * 255).permute([1, 2, 0]).cpu().numpy().astype(np.uint8).copy()
    if file == "45663149_63317.287.png":
        if not os.path.exists(save_path):
            os.mkdir(save_path)
        img = cv2.rectangle(np.array(img),(450,300),(650,500),(255,0,0),3)
        img2 = img[300:500,450:650,:]
        img2 = cv2.resize(img2,(600,600))
        img[-600:,-600:,:] = img2
        img = Image.fromarray(img)
        img.save(os.path.join(save_path,model_name))
    elif file == "47333452_54964.801.png":
        if not os.path.exists(save_path):
            os.mkdir(save_path)
        img = cv2.rectangle(img,(850,750),(1050,950),(255,0,0),3)
        img2 = img[750:950,850:1050,:]
        img2 = cv2.resize(img2,(600,600))
        img[:600,:600,:] = img2
        img = Image.fromarray(img)
        img.save(os.path.join(save_path,model_name))
    elif file == "45662975_26088.409.png":
        if not os.path.exists(save_path):
            os.mkdir(save_path)
        img = cv2.rectangle(img,(1200,500),(1400,700),(255,0,0),3)
        img2 = img[500:700,1200:1400,:]
        img2 = cv2.resize(img2,(600,600))
        img[:600,:600,:] = img2
        img = Image.fromarray(img)
        img.save(os.path.join(save_path,model_name))
    elif file == "47331653_23107.364.png":
        if not os.path.exists(save_path):
            os.mkdir(save_path)
        img = cv2.rectangle(img,(700,300),(900,500),(255,0,0),3)
        img2 = img[ 300:500,700:900,:]
        img2 = cv2.resize(img2,(600,600))
        img[-600:,-600:,:] = img2
        img = Image.fromarray(img)
        img.save(os.path.join(save_path,model_name))
    elif file == "47332908_12131.719.png":
        if not os.path.exists(save_path):
            os.mkdir(save_path)
        img = cv2.rectangle(img,(700,300),(900,500),(255,0,0),3)
        img2 = img[ 300:500,700:900,:]
        img2 = cv2.resize(img2,(600,600))
        img[-600:,-600:,:] = img2
        img = Image.fromarray(img)
        img.save(os.path.join(save_path,model_name))         
        

/home/ziyao/miniconda3/envs/deblur/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ziyao/miniconda3/envs/deblur/lib/python3.10/site-packages/torch/nn/modules/conv.py:459: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at /opt/conda/conda-bld/pytorch_1682343995026/work/aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv2d(input, weight, bias, self.stride,
